##### Setting up Search

In [4]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth.csv")

In [5]:
df_ground_truth.head()

,question,document,course
0,"If I miss the deadline for submissions, can I ...",74eb249bbf,llm-zoomcamp
1,What is the format of the project submissions ...,74eb249bbf,llm-zoomcamp
2,Can someone who has already joined the course ...,74eb249bbf,llm-zoomcamp
3,How long after submitting my project will I re...,74eb249bbf,llm-zoomcamp
4,Will there be any penalties or late fees for m...,74eb249bbf,llm-zoomcamp


In [6]:
ground_truth = df_ground_truth.to_dict(orient="records")
ground_truth[:5]

[{'question': 'If I miss the deadline for submissions, can I still get a certificate?',
  'document': '74eb249bbf',
  'course': 'llm-zoomcamp'},
 {'question': 'What is the format of the project submissions and what are the required materials?',
  'document': '74eb249bbf',
  'course': 'llm-zoomcamp'},
 {'question': 'Can someone who has already joined the course also submit their project later?',
  'document': '74eb249bbf',
  'course': 'llm-zoomcamp'},
 {'question': 'How long after submitting my project will I receive my certificate?',
  'document': '74eb249bbf',
  'course': 'llm-zoomcamp'},
 {'question': 'Will there be any penalties or late fees for me if I miss the submission deadline?',
  'document': '74eb249bbf',
  'course': 'llm-zoomcamp'}]

In [7]:
# Load the documents and build a minsearch index
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc['course'] == 'llm-zoomcamp':
        documents_llm.append(doc)
        
documents = documents_llm
index = build_index(documents)

In [9]:
boost = {"question": 3.0, "section": 0.5}

index.search("What is the course about?", num_results=5, boost_dict=boost)

[{'id': 'db78580409',
  'course': 'llm-zoomcamp',
  'section': 'Module 2: Vector Search',
  'question': 'What is the cosine similarity?',
  'answer': 'Cosine similarity is a measure used to calculate the similarity between two non-zero vectors, often used in text analysis to determine how similar two documents are based on their content. This metric computes the cosine of the angle between two vectors, which are typically word counts or TF-IDF values of the documents. The cosine similarity value ranges from -1 to 1, where 1 indicates that the vectors are identical, 0 indicates that the vectors are orthogonal (no similarity), and -1 represents completely opposite vectors.'},
 {'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](h

In [8]:
def text_search(query):
    boost_dict = {
        'question': 3.0,
        'section': 0.5,
    }
    
    return index.search(query, num_results=5, boost_dict=boost_dict)

##### Collecting relevance data

In [10]:
# Starting with one ground truth data
q = ground_truth[0]
q

{'question': 'If I miss the deadline for submissions, can I still get a certificate?',
 'document': '74eb249bbf',
 'course': 'llm-zoomcamp'}

In [11]:
# Run search for the question in the ground truth data
doc_id = q['document']
results = text_search(q['question'])

In [12]:
# Compare
for d in results:
    print(f'{d["id"]} == {doc_id} : {d["id"] == doc_id}')

9f689c185f == 74eb249bbf : False
cdc3b285e5 == 74eb249bbf : False
69d122f12e == 74eb249bbf : False
a9353fadfe == 74eb249bbf : False
74eb249bbf == 74eb249bbf : True


In [14]:
relevance = []
for d in results:
    relevance.append(int(d['id'] == doc_id))

relevance

[0, 0, 0, 0, 1]

In [15]:
# put into a function
def compute_relevance_text(q):
    doc_id = q['document']
    results = text_search(q['question'])
    
    relevance = []
    for d in results:
        relevance.append(int(d['id'] == doc_id))
        
    return relevance

In [16]:
# Examples
q = ground_truth[11]
print(q["question"])
compute_relevance_text(q)

How do I submit a question to Slido during live sessions? Is it through a URL or on the website?


[1, 0, 0, 0, 0]

In [17]:
# Do the same thing for all the ground truth data
from tqdm.auto import tqdm

def compute_relevance_all(ground_truth):
    relevance_all = []
    
    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_all.append(relevance)
        
    return relevance_all

In [18]:
# Call it for the first 15 ground truth data
ground_truth_subset = ground_truth[:15]
relevance_all = compute_relevance_all(ground_truth_subset)

  0%|          | 0/15 [00:00<?, ?it/s]

In [19]:
relevance_all

[[0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [20]:
# Make the relevance function generic to work with any search function
def compute_relevance(q, search_function):
    doc_id = q['document']
    results = search_function(q['question'])
    
    relevance = []
    for d in results:
        relevance.append(int(d['id'] == doc_id))
        
    return relevance

In [21]:
def compute_relevance_all(ground_truth, search_function):
    relevance_all = []
    
    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_all.append(relevance)
        
    return relevance_all

In [22]:
# Use it
relevance_all = compute_relevance_all(ground_truth_subset, text_search)
relevance_all

  0%|          | 0/15 [00:00<?, ?it/s]

[[0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [24]:
# Run for all ground truth data
relevance_all = compute_relevance_all(ground_truth, text_search)

relevance_all[:15]

  0%|          | 0/700 [00:00<?, ?it/s]

[[0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

#### Search Evaluation Metrics

In [25]:
sample = relevance_all[:15]

sample

[[0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [26]:
cnt = 0
for line in sample:
    if 1 in line:
        cnt += 1
        
cnt

9

In [27]:
# The hit rate is:
cnt / len(sample)

0.6

In [28]:
# put this logic into a function
def hit_rate(relevance):
    cnt = 0
    
    for line in relevance:
        if 1 in line:
            cnt += 1
            
    return cnt / len(relevance)

In [32]:
hit_rate(sample)

0.6

##### Mean Reciprocal Rank

In [33]:
total_score = 0.0

for line in sample:
    for rank in range(len(line)):
        if line[rank] == 1:
            total_score += 1 / (rank + 1)
            break

total_score

5.483333333333333

In [34]:
total_score / len(sample)

0.3655555555555556

In [35]:
# Put this logic into a function
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score += 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [36]:
mrr(sample)

0.3655555555555556

In [37]:
# Putting it together
def evaluate(ground_truth, search_function):
    relevance_all = compute_relevance_all(ground_truth, search_function)
    hit_rate_score = hit_rate(relevance_all)
    mrr_score = mrr(relevance_all)
    
    return {
        "hit_rate": hit_rate_score,
        "mrr": mrr_score
    }

In [38]:
# We can evaluate any search function
evaluate(ground_truth, text_search)

  0%|          | 0/700 [00:00<?, ?it/s]

{'hit_rate': 0.68, 'mrr': 0.5473333333333329}

In [39]:
def text_searchv2(query):
    boost_dict = {
        'question': 2.0,
        'section': 0.5,
    }
    
    return index.search(query, num_results=5, boost_dict=boost_dict)

In [40]:
evaluate(ground_truth, text_searchv2)

  0%|          | 0/700 [00:00<?, ?it/s]

{'hit_rate': 0.7157142857142857, 'mrr': 0.5837380952380948}

#### Search Parameter Tuning

In [41]:
def search_function_with_boost(query, question_boost=2.0):
    boost_dict = {
        'question': question_boost,
        'section': 0.5,
    }
    return index.search(query, num_results=5, boost_dict=boost_dict)

In [42]:
# Evaluate several boost values
for boost in [0.5, 1.0, 1.5, 2.0, 3.0, 10.0]:
    print(f"Evaluating with question boost: {boost}")
    results = evaluate(ground_truth, lambda q: search_function_with_boost(q, question_boost=boost))
    print(f"Results: {results}\n")

Evaluating with question boost: 0.5


  0%|          | 0/700 [00:00<?, ?it/s]

Results: {'hit_rate': 0.79, 'mrr': 0.6771904761904758}

Evaluating with question boost: 1.0


  0%|          | 0/700 [00:00<?, ?it/s]

Results: {'hit_rate': 0.7657142857142857, 'mrr': 0.6469999999999996}

Evaluating with question boost: 1.5


...

Results: {'hit_rate': 0.7157142857142857, 'mrr': 0.5837380952380948}

Evaluating with question boost: 3.0


  0%|          | 0/700 [00:00<?, ?it/s]

Results: {'hit_rate': 0.68, 'mrr': 0.5473333333333329}

Evaluating with question boost: 10.0


  0%|          | 0/700 [00:00<?, ?it/s]

Results: {'hit_rate': 0.6214285714285714, 'mrr': 0.48590476190476156}



In [43]:
# Boost for multiple fields
def search_function_with_multiple_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        'question': question_boost,
        'answer': answer_boost,
        'section': section_boost,
    }
    return index.search(query, num_results=5, boost_dict=boost_dict)

In [44]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(f"Evaluating with question boost: {question_boost}, answer boost: {answer_boost}, section boost: {section_boost}")
            res = evaluate(ground_truth, lambda q: search_function_with_multiple_boosts(q, question_boost, answer_boost, section_boost))
            results.append({
                "question_boost": question_boost,
                "answer_boost": answer_boost,
                "section_boost": section_boost,
                "hit_rate": res["hit_rate"],
                "mrr": res["mrr"]
            })
            print(f"Results: {res}\n")

Evaluating with question boost: 1.0, answer boost: 1.0, section boost: 0.1


  0%|          | 0/700 [00:00<?, ?it/s]

Results: {'hit_rate': 0.7957142857142857, 'mrr': 0.6722857142857139}

Evaluating with question boost: 1.0, answer boost: 1.0, section boost: 0.2


  0%|          | 0/700 [00:00<?, ?it/s]

Results: {'hit_rate': 0.7928571428571428, 'mrr': 0.6715714285714282}

Evaluating with question boost: 1.0, answer boost: 1.0, section boost: 0.5


...

Results: {'hit_rate': 0.8485714285714285, 'mrr': 0.7317857142857138}

Evaluating with question boost: 5.0, answer boost: 10.0, section boost: 0.2


  0%|          | 0/700 [00:00<?, ?it/s]

Results: {'hit_rate': 0.8485714285714285, 'mrr': 0.7312142857142854}

Evaluating with question boost: 5.0, answer boost: 10.0, section boost: 0.5


  0%|          | 0/700 [00:00<?, ?it/s]

Results: {'hit_rate': 0.8571428571428571, 'mrr': 0.7346190476190472}



In [45]:
# Sort by MRR score
df_results = pd.DataFrame(results)
df_results.sort_values(by="mrr", ascending=False).head(10)

,question_boost,answer_boost,section_boost,hit_rate,mrr
23,2.0,10.0,0.5,0.860000,0.739048
7,1.0,4.0,0.2,0.857143,0.738810
20,2.0,4.0,0.5,0.854286,0.735762
8,1.0,4.0,0.5,0.851429,0.735476
22,2.0,10.0,0.2,0.851429,0.735310
35,5.0,10.0,0.5,0.857143,0.734619
19,2.0,4.0,0.2,0.857143,0.734619
3,1.0,2.0,0.1,0.857143,0.734619
6,1.0,4.0,0.1,0.855714,0.734476
21,2.0,10.0,0.1,0.847143,0.734119
